In [ ]:
import cv2
import os
import json
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

class AnnotatorV3:
    def __init__(self, video_folder, json_input_folder, output_folder):
        self.video_folder = video_folder
        self.json_input_folder = json_input_folder
        self.output_folder = output_folder
        
        # Ensure output directory exists
        os.makedirs(self.output_folder, exist_ok=True)
        
        # Get Video Files
        self.video_files = [f for f in os.listdir(video_folder) if f.lower().endswith(('.mp4', '.mov', '.avi'))]
        self.video_files.sort()
        
        print(f"--- ANNOTATOR V3 INITIALIZED ---")
        print(f"🎥 Videos found: {len(self.video_files)}")
        print(f"📂 Reading existing JSONs from: {self.json_input_folder}")
        print(f"💾 Saving new JSONs to: {self.output_folder}")
        
        self.current_index = 0
        self.current_stage = "start" 
        self.temp_start_data = None
        
        # UI State
        self.ptr_x = 0; self.ptr_y = 0
        self.frame_h = 0; self.frame_w = 0
        self.orig_frame = None 
        self.collected_points = [] 
        
        self.setup_ui()
        
    def setup_ui(self):
        self.img_widget = widgets.Image(format='jpeg', width=800)
        
        self.step_size = widgets.ToggleButtons(
            options=[('Precise (5px)', 5), ('Fast (50px)', 50), ('Jump (150px)', 150)],
            value=50, description='Speed:', style={'button_width': '100px'}
        )
        
        btn_layout = widgets.Layout(width='60px', height='60px')
        b_up = widgets.Button(description='▲', layout=btn_layout)
        b_down = widgets.Button(description='▼', layout=btn_layout)
        b_left = widgets.Button(description='◀', layout=btn_layout)
        b_right = widgets.Button(description='▶', layout=btn_layout)
        b_confirm = widgets.Button(description='🎯 Confirm', button_style='success', layout=widgets.Layout(width='150px'))
        b_skip = widgets.Button(description='Skip >>', layout=widgets.Layout(width='100px'))
        
        self.lbl_status = widgets.Label(value="Initializing...")
        
        b_up.on_click(lambda b: self.move_pointer(0, -1))
        b_down.on_click(lambda b: self.move_pointer(0, 1))
        b_left.on_click(lambda b: self.move_pointer(-1, 0))
        b_right.on_click(lambda b: self.move_pointer(1, 0))
        b_confirm.on_click(self.confirm_point)
        b_skip.on_click(self.skip_video)
        
        dpad = widgets.VBox([
            widgets.HBox([widgets.Label(layout=btn_layout), b_up, widgets.Label(layout=btn_layout)]),
            widgets.HBox([b_left, widgets.Label(layout=btn_layout), b_right]),
            widgets.HBox([widgets.Label(layout=btn_layout), b_down, widgets.Label(layout=btn_layout)])
        ])
        
        self.ui = widgets.VBox([
            self.lbl_status,
            self.img_widget, 
            widgets.HBox([dpad, widgets.VBox([self.step_size, b_confirm, b_skip])])
        ])

    def start(self):
        display(self.ui)
        self.load_video()

    def load_video(self):
        if self.current_index >= len(self.video_files):
            self.lbl_status.value = "🎉 ALL VIDEOS COMPLETED."
            self.img_widget.value = b''
            return

        filename = self.video_files[self.current_index]
        base_name = os.path.splitext(filename)[0]
        json_filename = f"{base_name}_ground_truth.json"
        
        # --- PATH LOGIC ---
        # 1. Check Output Folder (Highest Priority - Most recent work)
        path_out = os.path.join(self.output_folder, json_filename)
        # 2. Check Input Folder (Secondary Priority - Archive/Old work)
        path_in = os.path.join(self.json_input_folder, json_filename)
        
        found_data = None
        source_used = "None"

        if os.path.exists(path_out):
            with open(path_out, 'r') as f: found_data = json.load(f)
            source_used = "Output Folder"
        elif os.path.exists(path_in):
            with open(path_in, 'r') as f: found_data = json.load(f)
            source_used = "Input Folder"

        # --- RESUME LOGIC ---
        if found_data:
            # Case A: Fully Done
            if "end_frame_players" in found_data:
                print(f"Skipping {filename} (Found complete data in {source_used})")
                self.current_index += 1
                self.load_video()
                return
            
            # Case B: Partial Data (Resume)
            elif "player_1" in found_data and self.current_stage == "start":
                print(f"Resuming {filename} (Found partial data in {source_used}) -> Jumping to END frame")
                self.temp_start_data = {
                    "frame_idx": found_data.get("frame_number", 20),
                    "players": {"p1": found_data["player_1"], "p2": found_data["player_2"]}
                }
                self.current_stage = "end"
                # Continue execution to load video frame...

        # --- LOAD VIDEO FRAME ---
        vid_path = os.path.join(self.video_folder, filename)
        cap = cv2.VideoCapture(vid_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if self.current_stage == "start":
            target = 20
        else:
            target = max(0, total_frames - 20)
            
        cap.set(cv2.CAP_PROP_POS_FRAMES, target)
        ret, frame = cap.read()
        cap.release()
        
        if not ret:
            print(f"❌ Error reading {filename}. Skipping.")
            self.current_index += 1
            self.current_stage = "start"
            self.load_video()
            return

        self.orig_frame = frame
        self.frame_h, self.frame_w, _ = frame.shape
        self.ptr_x = self.frame_w // 2
        self.ptr_y = self.frame_h // 2
        self.collected_points = []
        self.target_frame_idx = target
        
        self.update_display()
        label = "START (Frame 20)" if self.current_stage == "start" else f"END (Frame {target})"
        self.lbl_status.value = f"{filename} | {label} | Click PLAYER 1"

    def move_pointer(self, dx, dy):
        step = self.step_size.value
        self.ptr_x = max(0, min(self.frame_w, self.ptr_x + (dx * step)))
        self.ptr_y = max(0, min(self.frame_h, self.ptr_y + (dy * step)))
        self.update_display()

    def update_display(self):
        disp = self.orig_frame.copy()
        for pt in self.collected_points:
            cv2.circle(disp, (pt[0], pt[1]), 10, (0, 255, 0), -1) 
        cv2.line(disp, (self.ptr_x - 20, self.ptr_y), (self.ptr_x + 20, self.ptr_y), (0, 0, 255), 2)
        cv2.line(disp, (self.ptr_x, self.ptr_y - 20), (self.ptr_x, self.ptr_y + 20), (0, 0, 255), 2)
        _, enc = cv2.imencode('.jpg', disp)
        self.img_widget.value = enc.tobytes()

    def confirm_point(self, b):
        self.collected_points.append([self.ptr_x, self.ptr_y])
        if len(self.collected_points) == 1:
            self.ptr_y = max(50, self.ptr_y - 200)
            self.update_display()
            self.lbl_status.value = "✅ P1 Saved. Move to PLAYER 2"
        elif len(self.collected_points) == 2:
            self.finish_stage()

    def finish_stage(self):
        points = {"p1": self.collected_points[0], "p2": self.collected_points[1]}
        if self.current_stage == "start":
            self.temp_start_data = {"frame_idx": self.target_frame_idx, "players": points}
            self.current_stage = "end"
            self.load_video()
        else:
            self.save(points)
            self.current_stage = "start"
            self.current_index += 1
            self.load_video()

    def save(self, end_points):
        filename = self.video_files[self.current_index]
        # Always save to Output Folder
        path = os.path.join(self.output_folder, os.path.splitext(filename)[0] + "_ground_truth.json")
        
        data = {
            "video_file": filename,
            "start_frame": self.temp_start_data["frame_idx"],
            "start_frame_players": self.temp_start_data["players"],
            "end_frame": self.target_frame_idx,
            "end_frame_players": end_points
        }
        with open(path, 'w') as f: json.dump(data, f, indent=4)
        print(f"✅ Saved: {path}")

    def skip_video(self, b):
        self.current_index += 1
        self.current_stage = "start"
        self.load_video()

In [ ]:
# 1. Path to VIDEOS (e.g., .mp4 files)
VIDEO_DIR = '/kaggle/input/tennis-rally-videos'

# 2. Path to EXISTING JSONS (Where you saved them before)
OLD_JSON_DIR = '/kaggle/input/tennis-rally-videos/detection_ground_truth' 

# 3. Path to SAVE NEW JSONS
NEW_OUTPUT_DIR = '/kaggle/working/player_ground_truths'

# Initialize and Start
app = AnnotatorV3(
    video_folder=VIDEO_DIR, 
    json_input_folder=OLD_JSON_DIR, 
    output_folder=NEW_OUTPUT_DIR
)
app.start()